<a href="https://colab.research.google.com/github/Sarkis55/CA04---Ensemble_Learning_Census_Data/blob/main/CA04_Ensemble_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installing sweetviz and Autoviz
pip install sweetviz

In [ ]:
#Libaries for Preprocessing and DQA
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import sweetviz as sv

from sklearn.preprocessing import LabelEncoder

#Libraries needed for training, testing and evaluating model
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
#Retrieving data and loading them into dataframe
csv_url = 'https://github.com/ArinB/MSBA-CA-03-Decision-Trees/blob/master/census_data.csv?raw=true'
df = pd.read_csv(csv_url)

display(df.head())

## Preprocessing and DQA

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
#Generating a EDA report using python's sweetviz library
report = sv.analyze(df)
report.show_html()

In [ ]:
#Checking for Missing Data
df.isnull().sum()

In [ ]:
#Getting a value count for each column
for col in df.columns:
  print(f"Value counts for column: {col}")
  print(df[col].value_counts())
  print("\n")

In [ ]:
#Discretizing features using label encoder.
#Since the majority of the values are object we use encoder to reprsent each value as an integer.

LE = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
      df[col] = LE.fit_transform(df[col])

print("DataFrame after encoding all object columns:")
display(df.head())

#Source: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html

In [ ]:
#Getting a value count for each column after encoding
for col in df.columns:
  print(f"Value counts for column: {col}")
  print(df[col].value_counts())
  print("\n")


In [ ]:
#Heatmap to see feature coorelation
sns.heatmap(df.corr(), annot=True)

In [ ]:
#dropping education_bin since it is essentially the same as education_num_bin
df = df.drop('education_bin', axis=1)

In [ ]:
#Since we have a column that labels what rows are training and testing, we manually split them by creating two separate dataframes.

#Whatever that is labeled "train" goes to the training dataframe and same for testing
train_df = df[df['flag'] == 1]
test_df = df[df['flag'] == 0]

In [ ]:
# Separate the features (X) and target (y).
# We then drop the "flag" column so that the model doesn't cheat by associating a pattern based on a speciifc value
X_train = train_df.drop(['y', 'flag'], axis=1)
y_train = train_df['y']

X_test = test_df.drop(['y', 'flag'], axis=1)
y_test = test_df['y']

#Code generated by Gemini

## Building Randome Forest Model, then finding optimal value of n_estimators

Training the data on a random forest model, then we compare the Accuracy and AUC among different values for the 'n_estimator' hyper parameter, in order to find a value that is optimal.

In [ ]:
accuracy_results = []
auc_results = []
n_estimator_options = [50,100,150,200,250,300,350,400,450,500]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

for n_estimator in n_estimator_options:
  #Training and testing random forest model with each value for 'n_estimator'
  model = RandomForestClassifier(n_estimators = n_estimator, random_state = 101)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  #getting the accuracy and auc score
  accuracy = accuracy_score(y_test, y_pred)
  auc = roc_auc_score(y_test, y_pred)

  #Storing the results for accuracy and auc after each iteration
  accuracy_results.append(accuracy)
  auc_results.append(float(auc))

print("Accuracy Results:", accuracy_results)
print("AUC Results:", auc_results)

In [ ]:
#Plotting line graphs for Accuracy vs.n_estimator, and AUC Vs.n_estimator

fig, (ax1, ax2) = plt.subplots(2)

ax1.plot(n_estimator_options, accuracy_results)
ax1.set_xlabel('n_estimators')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs. n_estimator')

ax2.plot(n_estimator_options, auc_results)
ax2.set_xlabel('n_estimators')
ax2.set_ylabel('AUC')
ax2.set_title('AUC vs. n_estimator')

plt.tight_layout()
plt.show()

#Storing best random forest accuacy and auc scores
best_rf_accuracy = max(accuracy_results)
best_rf_auc = max(auc_results)

#Printing best accuracy and auc and its respective estimator value
print("Optimal estimator Random Forest Accuracy: (", n_estimator_options[accuracy_results.index(best_rf_accuracy)], ',', best_rf_accuracy, ")")
print("Optimal estimator and Random Forest AUC: (", n_estimator_options[auc_results.index(best_rf_auc)], ',', best_rf_auc, ")")

#### 1. Write your observations about the Classifier’s behavior with respect to the number of estimators.
*   Based on the graphs, the values for both AUC and accuracy seems to flucuate as the number of estimators inceases, but at a very slight variation of less than 0.1%. Therefore the performance of the random forest classifier seems to plateau if we keep all other hyperparameters as default values.

#### 2. Is there an optimal value of the estimator within the given range?
*   Based on the two graphs, the optimal value of the estimator is 50, which gives an accuracy of 0.838 and AUC of 0.748.



## Building Gradient Boost Model, then recording optimal estimator